# Haiku 4.5 -- Tree-of-Thoughts (BFS-2, depth=3)

**Sync** implementation. This experiment is **NOT batched** because the
branching state (each beam path's accumulated reasoning chain) must be built
sequentially -- the implementation complexity plus custom_id mapping bug
risk outweighs the marginal cost savings of batching.

**Model:** `claude-haiku-4-5-20251001`
**Algorithm:** BFS-2, depth=3, sort by sure/maybe/impossible -> top-2 beam
**Final extraction:** feed the full reasoning trace back to the model and ask for `#### N`
**Seed:** 42 -> identical 100 problems to the Llama 8B runs

## Cost and time estimate
- ~21 API calls per problem
- 100 problems: ~2100 total calls
- Haiku 4.5 sync price: $1/M input, $5/M output (no batch discount on sync ToT)
- Estimated cost: ~$1.0-1.5
- Tier 1 (50 RPM, 0.3s sleep): ~30-60 min

## Prerequisites
1. **Secrets:** add `ANTHROPIC_API_KEY`
2. **Drive:** upload `MyDrive/NLP_Haiku/data/gsm8k_test.json`

## Output
- `MyDrive/NLP_Haiku/results/tot_haiku_d3.json` (checkpoint written after every problem)


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE    = '/content/drive/MyDrive/NLP_Haiku'
DRIVE_DATA    = os.path.join(DRIVE_BASE, 'data')
DRIVE_RESULTS = os.path.join(DRIVE_BASE, 'results')

os.makedirs(DRIVE_DATA,    exist_ok=True)
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print('Drive mounted.')
print('Data dir   :', DRIVE_DATA)
print('Results dir:', DRIVE_RESULTS)

In [ ]:
# 2. Install the Anthropic SDK
!pip install -q anthropic
import anthropic
print('anthropic', anthropic.__version__)

In [ ]:
# 3. Load API key from Colab Secrets
from google.colab import userdata
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY').strip()
if not ANTHROPIC_API_KEY:
    raise ValueError('ANTHROPIC_API_KEY not found. Add it from the Secrets panel.')
print('API key loaded.')

In [ ]:
# 4. Configuration
MODEL       = 'claude-haiku-4-5-20251001'
N_SAMPLES   = 100
SEED        = 42
DEPTH       = 3      # this notebook: depth=3
BRANCHING   = 2
MAX_TOKENS  = 1024

DATA_FILE       = os.path.join(DRIVE_DATA,    'gsm8k_test.json')
CHECKPOINT_FILE = os.path.join(DRIVE_RESULTS, f'tot_haiku_d{DEPTH}.json')

print(f'Model      : {MODEL}')
print(f'N samples  : {N_SAMPLES}')
print(f'Depth      : {DEPTH}')
print(f'Branching  : {BRANCHING}')
print(f'Checkpoint : {CHECKPOINT_FILE}')

In [ ]:
# 5. Load GSM8K test data (seed=42)
import json
import random

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(
        f'Data file not found: {DATA_FILE}\n'
        'Please upload gsm8k_test.json to Drive/NLP_Haiku/data/.'
    )

with open(DATA_FILE, encoding='utf-8') as f:
    all_data = json.load(f)

random.seed(SEED)
test_data = random.sample(all_data, N_SAMPLES)

print(f'Total test set: {len(all_data)}')
print(f'This run      : {len(test_data)} problems (seed={SEED})')

In [ ]:
# 6. LLM client (Anthropic) + retry + parser
import time
import re
from anthropic import Anthropic
from anthropic import APIStatusError, APIConnectionError, RateLimitError

client = Anthropic(api_key=ANTHROPIC_API_KEY, timeout=120.0)

# Tier 1 limit is 50 RPM; sleep a little after each call to avoid 429s
SLEEP_AFTER_CALL = 0.3   # seconds; raise if you hit rate limit pressure

# Connectivity smoke test
_test = client.messages.create(
    model=MODEL,
    max_tokens=10,
    messages=[{'role': 'user', 'content': 'Say: hello'}],
)
_test_text = next((b.text for b in _test.content if getattr(b, 'type', None) == 'text'), '')
print('Connectivity test:', _test_text)


def chat(prompt, temperature=0.0, max_tokens=MAX_TOKENS):
    for attempt in range(8):
        try:
            resp = client.messages.create(
                model=MODEL,
                max_tokens=max_tokens,
                temperature=temperature,
                messages=[{'role': 'user', 'content': prompt}],
            )
            text = next((b.text for b in resp.content if getattr(b, 'type', None) == 'text'), '')
            time.sleep(SLEEP_AFTER_CALL)
            return text.strip(), resp.usage.input_tokens, resp.usage.output_tokens
        except RateLimitError:
            wait = 30 * (attempt + 1)
            print(f'  Rate limit, waiting {wait}s...')
            time.sleep(wait)
        except (APIConnectionError, APIStatusError) as e:
            wait = 10 * (attempt + 1)
            print(f'  API error ({type(e).__name__}), waiting {wait}s...')
            time.sleep(wait)
    raise RuntimeError('chat() failed after 8 attempts.')


def extract_answer(text):
    if not text:
        return None
    m = re.search(r'####\s*\$?([\d,]+\.?\d*)', text)
    if m:
        try: return float(m.group(1).replace(',', ''))
        except: pass
    matches = re.findall(
        r'(?:final\s+)?answer\s+is\s*[:\-]?\s*\$?([\d,]+\.?\d*)',
        text, re.IGNORECASE
    )
    if matches:
        try: return float(matches[-1].replace(',', ''))
        except: pass
    nums = re.findall(r'[\d,]+\.?\d*', text.replace(',', ''))
    return float(nums[-1]) if nums else None


def gold_answer(text):
    m = re.search(r'####\s*([\d,\.]+)', text)
    return float(m.group(1).replace(',', '')) if m else None

print('LLM client and helpers ready.')

In [ ]:
# 7. ToT prompt templates (identical to run_tot_fixed.py)
THOUGHT_PROMPT = """Solve this math problem step by step. Generate the next single reasoning step only.

Problem: {problem}
Steps so far: {steps}

Next step:"""

EVALUATE_PROMPT = """Problem: {problem}
Reasoning so far: {steps}
Candidate next step: {candidate}

Is this step leading toward the correct solution? Reply with one word: sure, maybe, or impossible."""

EXTRACT_PROMPT = """Problem: {problem}
Reasoning:
{reasoning}

Based on the reasoning above, what is the final numeric answer?
Write only the number on the last line in this exact format:
#### <number>"""

print('Prompt templates ready.')

In [ ]:
# 8. ToT solver (BFS-2)
def solve(problem):
    beam = ['']
    in_tok = out_tok = 0

    for level in range(DEPTH):
        candidates = []
        for path in beam:
            for _ in range(BRANCHING):
                thought, it1, ot1 = chat(
                    THOUGHT_PROMPT.format(problem=problem, steps=path),
                    temperature=0.7,
                )
                score, it2, ot2 = chat(
                    EVALUATE_PROMPT.format(problem=problem, steps=path, candidate=thought),
                    temperature=0.0,
                )
                in_tok  += it1 + it2
                out_tok += ot1 + ot2
                candidates.append((path + '\n' + thought, score))

        order = {'sure': 0, 'maybe': 1, 'impossible': 2}
        candidates.sort(key=lambda x: order.get(x[1].strip().lower().split()[0] if x[1].strip() else 'maybe', 3))
        beam = [c[0] for c in candidates[:BRANCHING]]

    best_path = beam[0]

    # Final extraction step
    final_resp, it3, ot3 = chat(
        EXTRACT_PROMPT.format(problem=problem, reasoning=best_path),
        temperature=0.0,
    )
    in_tok  += it3
    out_tok += ot3

    predicted = extract_answer(final_resp)
    if predicted is None:
        predicted = extract_answer(best_path)

    return {
        'reasoning':      best_path,
        'final_response': final_resp,
        'predicted':      predicted,
        'usage':          {'input_tokens': in_tok, 'output_tokens': out_tok},
    }

print('Solver ready.')

In [ ]:
# 9. Run + checkpoint (write to Drive after every problem; safe to interrupt)
results        = []
done_questions = set()

if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, encoding='utf-8') as f:
        ckpt = json.load(f)
    results        = ckpt.get('results', [])
    done_questions = {r['question'] for r in results}
    print(f'Checkpoint found: {len(results)} problems already done. Acc={ckpt.get("accuracy")}%')
else:
    print('No checkpoint found; starting from scratch.')

total = len(test_data)

for i, item in enumerate(test_data, 1):
    if item['question'] in done_questions:
        print(f'[{i}/{total}] skipping (already done)', end='\r', flush=True)
        continue

    print(f'[{i}/{total}] solving...', end='\r', flush=True)

    out  = solve(item['question'])
    gold = gold_answer(item['answer'])

    results.append({
        'question':       item['question'],
        'gold':           gold,
        'predicted':      out['predicted'],
        'correct':        out['predicted'] == gold,
        'reasoning':      out['reasoning'],
        'final_response': out['final_response'],
        'usage':          out['usage'],
    })

    correct_so_far = sum(r['correct'] for r in results)
    with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
        json.dump({
            'strategy':  f'tot_haiku_depth{DEPTH}_branch{BRANCHING}',
            'model':     MODEL,
            'depth':     DEPTH,
            'branching': BRANCHING,
            'n_done':    len(results),
            'accuracy':  round(correct_so_far / len(results) * 100, 1),
            'results':   results,
        }, f, indent=2, ensure_ascii=False)

correct = sum(r['correct'] for r in results)
n       = len(results)
print(f'\nToT (depth={DEPTH}) finished: {correct}/{n} = {correct/n*100:.1f}%')

In [ ]:
# 10. Summary + carnival gold-correction + Cobbe stratified

def n_ops(gold_text):
    pre = gold_text.split('####')[0]
    return pre.count('=')

# Cobbe stratified
buckets = {'easy': [], 'medium': [], 'hard': []}
for r, item in zip(results, test_data):
    ops = n_ops(item['answer'])
    if ops < 3:    buckets['easy'].append(r['correct'])
    elif ops == 3: buckets['medium'].append(r['correct'])
    else:          buckets['hard'].append(r['correct'])

raw = sum(r['correct'] for r in results)
gc  = raw
for r in results:
    if 'carnival' in r.get('question', '').lower() and not r.get('correct'):
        if r.get('predicted') == 2180.0:
            gc += 1

print(f'=== ToT (depth={DEPTH}) Haiku 4.5 summary ===')
print(f'Raw            : {raw}/{n} = {raw/n*100:.1f}%')
print(f'Gold-corrected : {gc}/{n} = {gc/n*100:.1f}%')
print(f'Easy           : {sum(buckets["easy"])}/{len(buckets["easy"])} = {100*sum(buckets["easy"])/max(1,len(buckets["easy"])):.1f}%')
print(f'Medium         : {sum(buckets["medium"])}/{len(buckets["medium"])} = {100*sum(buckets["medium"])/max(1,len(buckets["medium"])):.1f}%')
print(f'Hard           : {sum(buckets["hard"])}/{len(buckets["hard"])} = {100*sum(buckets["hard"])/max(1,len(buckets["hard"])):.1f}%')

# Token usage / cost (Haiku 4.5 sync full price: $1/M in, $5/M out)
total_in  = sum((r.get('usage', {}) or {}).get('input_tokens')  or 0 for r in results)
total_out = sum((r.get('usage', {}) or {}).get('output_tokens') or 0 for r in results)
COST = (total_in / 1e6) * 1.00 + (total_out / 1e6) * 5.00
print(f'\nToken usage:')
print(f'  Input  : {total_in:>10,}')
print(f'  Output : {total_out:>10,}')
print(f'  Total  : {total_in + total_out:>10,}')
print(f'\nEstimated cost (SYNC, $1/M in, $5/M out): ${COST:.4f}')

print(f'\n=== Llama 3.1 8B reference (side-by-side comparison) ===')
if DEPTH == 3:
    print('  ToT d=3 Llama: 81.0%  (100/100, gold-corrected)')
elif DEPTH == 6:
    print('  ToT d=6 Llama: 85.0%  (100/100, gold-corrected)')